# 2D analysis of resampled Einstein Toolkit data

Interactive exploration of one simulation, fitting the three-step workflow:

1. **Resample** the raw simulation output once with `../Resample/resample_2d.py`
   (per-variable HDF5 files: fast, lazy, small).
2. **Explore here**: scalar time series, 2D slices, computations, plotting styles.
3. **Render the movie** with `../Frames/make_frames.py` using the config YAML
   written at the bottom of this notebook.

The notebook *imports* `make_frames` and uses its `Fields` reader and
`render_frame()` directly, so every preview is pixel-identical to the frames
the movie will contain.

In [ ]:
%matplotlib inline

import os
import sys
import time

import numpy as np
import yaml
import h5py

import matplotlib.pyplot as plt
import matplotlib.colors as colors

from IPython.display import Image, display

# Reuse the frame-rendering infrastructure (Fields, render_frame, constants).
sys.path.insert(0, os.path.abspath("../Frames"))
import make_frames as M

## Configuration

One simulation per notebook session. `SIMDIR` (raw output) is only needed for
the scalar section; everything 2D reads the resampled HDF5 files.

In [ ]:
# Raw simulation directory (only used for scalar time series; optional).
SIMDIR = "/mnt/raarchive/chabanov/direct_Urca_runs/dU_10_15_linear_HR"

# Resampled 2D data (output of ../Resample). Absolute paths so the config
# YAML written below works no matter where make_frames.py runs.
DATA_DIR = os.path.abspath("../Frames/data")
LABEL = "example"

FIELDS = {
    "rho": os.path.join(DATA_DIR, "rho_b__%s.h5" % LABEL),
    # add more variables here, e.g.
    # "pi": os.path.join(DATA_DIR, "pi__%s.h5" % LABEL),
    # "P":  os.path.join(DATA_DIR, "P__%s.h5" % LABEL),
}

## Constants and unit conversions

Shared constants are taken **from `make_frames`** (single source of truth);
analysis-specific ones are defined here with their derivations. Everything the
movie step needs later travels through the config YAML, so nothing has to be
edited in two places.

In [ ]:
# --- shared with make_frames (do not redefine!) -----------------------------
MSUN_TO_KM = M.MSUN_TO_KM   # GM_sun/c^2 in km: code length unit -> km
MILLIS = M.MILLIS           # 1 ms in code time units (= 1e-3 * c[km/s] / MSUN_TO_KM)

# closure check: the two constants must be consistent
assert abs(MILLIS - 1e-3 * 299792.458 / MSUN_TO_KM) < 1e-9

# --- analysis-specific -------------------------------------------------------
T_MERGE = 0.0               # merger time in code units (0 for the example data)

# rest-mass density: 1 code unit = 1/FACTOR g/cm^3 (~6.18e17 g/cm^3)
FACTOR = 1.61887093132742e-18
RHO_SAT = 2.7e14 * FACTOR   # nuclear saturation density in code units

# temperature: stored in units of the neutron mass; x CONVERT_TEMP -> MeV
# CONVERT_TEMP = m_n c^2 = 939.565 MeV  (kB cancels in the derivation)
KB_OVER_MN = 8.617332478e-5 / 939.565e6
CONVERT_TEMP = (1.0 / KB_OVER_MN) * 8.617332478e-11
assert abs(CONVERT_TEMP - 939.565) < 1e-9

# density contour levels (code units); cgs values printed for reference
RHO_LEVELS = [1.828929564024961e-06, 0.00073157182561, 0.0013]
print("RHO_LEVELS in 10^15 g/cm^3:", [round(l / FACTOR / 1e15, 4) for l in RHO_LEVELS])

## A) Scalar time series (optional - needs the raw simulation)

Maxima/minima over the grid as functions of time, via postcactus. Skipped
automatically when `SIMDIR` is not reachable from this machine.

In [ ]:
HAVE_RAW_SIM = os.path.isdir(SIMDIR)

scalars = {}
if HAVE_RAW_SIM:
    from postcactus.simdir import SimDir
    sd = SimDir(SIMDIR)
    for var in ["rho_b", "temp"]:
        scalars[var + "_max"] = sd.ts.max[var]
    for var in ["alp"]:
        scalars[var + "_min"] = sd.ts.min[var]
    print("loaded:", ", ".join(scalars))
else:
    print("raw simulation not reachable (%s) - skipping scalar section" % SIMDIR)

In [ ]:
if scalars:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, (name, log_scale) in zip(
            axes, [("rho_b_max", False), ("temp_max", True), ("alp_min", False)]):
        s = scalars[name]
        ax.plot((s.t - T_MERGE) / MILLIS, s.y)
        ax.set_xlabel(r"$t - t_\mathrm{mer}$ [ms]")
        ax.set_title(name)
        if log_scale:
            ax.set_yscale("log")
    fig.tight_layout()

## B) 2D data: lazy HDF5 access

`M.Fields` opens the resampled files; slices are only read when indexed, so a
session costs megabytes, not gigabytes. Iterations and physical times come
straight from the files - no more manual bookkeeping or hardcoded lengths.

In [ ]:
fields = M.Fields(FIELDS)

iterations = fields.h5["rho"]["iterations"][:]
times = fields.h5["rho"]["times"][:]

print("frames     :", fields.n_frames())
print("iterations :", iterations[0], "...", iterations[-1])
print("times      : %.1f ... %.1f M  (%.2f ... %.2f ms)"
      % (times[0], times[-1], times[0] / MILLIS, times[-1] / MILLIS))

In [ ]:
def frame_at_time(t_ms, ref="rho"):
    # frame index whose time is closest to t_ms (relative to T_MERGE)
    t = fields.h5[ref]["times"][:]
    idx = int(np.searchsorted(t, T_MERGE + t_ms * MILLIS))
    return min(max(idx, 0), len(t) - 1)

idx = frame_at_time(2.0)
print("t = 2.0 ms -> frame", idx, "(it %d, t = %.3f ms)"
      % (fields.iteration("rho", idx), fields.time("rho", idx) / MILLIS))

## C) Quick look / playground

Free-form cell for trying computations and styles on a single slice -
anything `f.data(name, i)` based works (ratios, logs, differences, ...).

In [ ]:
idx = frame_at_time(2.0)
data = fields.data("rho", idx)                 # <- experiment here
x, y = fields.coords("rho")
X, Y = np.meshgrid(x * MSUN_TO_KM, y * MSUN_TO_KM, indexing="ij")

fig, ax = plt.subplots(figsize=(6, 5))
mesh = ax.pcolormesh(X, Y, data, norm=colors.LogNorm(vmin=1e-10, vmax=2e-3),
                     cmap="magma", shading="nearest")
ax.contour(X, Y, data, RHO_LEVELS, colors="black", linewidths=1.0)
ax.set_aspect("equal")
fig.colorbar(mesh, ax=ax)
ax.set_title("quick look: rho at t = %.2f ms" % (fields.time("rho", idx) / MILLIS));

## D) Panel definitions (the part shared with `make_frames.py`)

> **What must stay in sync between this notebook and `make_frames.py`:**
> the `PANELS` list below - the data lambdas, norms, colormaps and contour
> specs. It cannot travel through YAML (lambdas are code), so when a figure is
> final, **copy these panel dicts into `build_panels()` in
> `../Frames/make_frames.py`**. Everything else (paths, units, frame
> selection, output options, figure layout, contour levels - the full list is
> `M.CONFIGURABLE`) is written to the config YAML at the bottom and read by
> `make_frames.py` automatically, so it never needs manual syncing.

In [ ]:
movie_settings = {
    # input + units
    "FIELDS": FIELDS,
    "MSUN_TO_KM": MSUN_TO_KM,
    "MILLIS": MILLIS,
    "T_MERGE": T_MERGE,
    # frame selection
    "FRAME_START": 0,
    "FRAME_STOP": None,
    "FRAME_STRIDE": 1,
    # output
    "OUT_DIR": "./frames",
    "OUT_PREFIX": "rho_b",
    "OUT_EXT": "png",
    "ZFILL": 4,
    "MAKE_MOVIE": True,
    "FRAMERATE": 25,
    "MOVIE_NAME": "rho_b.mp4",
    # figure layout / style
    "FIG_ROWS": 1,
    "FIG_COLS": 1,
    "FIGSIZE": [7.5, 6.0],
    "DPI": 200,
    "FONT_SIZE": 15,
    "SUPTITLE": None,
    "XLABEL": r"$x~[\mathrm{km}]$",
    "YLABEL": r"$y~[\mathrm{km}]$",
    "XLIM": [-73.8, 73.8],
    "YLIM": [-73.8, 73.8],
    "SUBPLOTS_ADJUST": dict(left=0.12, right=0.95, bottom=0.085, top=0.95,
                            wspace=0.25, hspace=0.20),
    "RCPARAMS": {},
    # contour specs
    "RHO_LEVELS": RHO_LEVELS,
    "RHO_COLORS": ["black", "black", "black"],
    "RHO_STYLES": ["dashed", "solid", "dotted"],
    "RHO_LW": 1.3,
}
assert set(movie_settings) == set(M.CONFIGURABLE), "settings out of sync!"

# apply to the imported module so previews use exactly these values
for key, value in movie_settings.items():
    setattr(M, key, value)

# the shared contract: copy this into build_panels() in make_frames.py when final
PANELS = [
    {
        "title": r"$\rho;~t=$ {t:.3f} ms",
        "ref": "rho",
        "data": lambda f, i: f.data("rho", i),
        "cmap": "magma",
        "norm": colors.LogNorm(vmin=1e-10, vmax=2e-3),
        "time_offset": T_MERGE,
        "colorbar": True,
        "contours": [
            {"data": lambda f, i: f.data("rho", i), "ref": "rho",
             "levels": movie_settings["RHO_LEVELS"],
             "colors": movie_settings["RHO_COLORS"],
             "linestyles": movie_settings["RHO_STYLES"],
             "linewidths": movie_settings["RHO_LW"]},
        ],
    },
]
M.PANELS = PANELS
print("panels:", len(PANELS))

### Preview a movie frame

This calls the *real* `render_frame()` - what you see is exactly what the
movie will contain.

In [ ]:
M.OUT_DIR = "./_preview"
os.makedirs(M.OUT_DIR, exist_ok=True)

path = M.render_frame(fields, frame_at_time(2.0), 0)
display(Image(filename=path, width=560))

## E) Publication figure

Same mechanism, same style - just rendered to PDF at print DPI. (This is the
"simpler version": one `render_frame` call instead of hand-built panel
blocks; for a multi-panel publication figure, extend `PANELS` and
`FIG_ROWS`/`FIG_COLS` above.)

In [ ]:
M.OUT_EXT, M.DPI, M.OUT_PREFIX = "pdf", 300, "publication"
pdf_path = M.render_frame(fields, frame_at_time(2.0), 0)
M.OUT_EXT, M.DPI, M.OUT_PREFIX = (movie_settings["OUT_EXT"],
                                  movie_settings["DPI"],
                                  movie_settings["OUT_PREFIX"])
print("saved", pdf_path)

## F) Efficiency check

The old workflow unpickled *entire* variables (all iterations) before touching
anything. With lazy HDF5 the cost of looking at one frame is one slice:

In [ ]:
size_mb = os.path.getsize(FIELDS["rho"]) / 1e6

t0 = time.perf_counter()
_ = fields.data("rho", fields.n_frames() // 2)
t_one = time.perf_counter() - t0

t0 = time.perf_counter()
_all = fields.h5["rho"]["data"][:]          # what eager loading would cost
t_full = time.perf_counter() - t0
mem_mb = _all.nbytes / 1e6
del _all

print("file size                 : %7.1f MB" % size_mb)
print("read ONE frame (lazy)     : %7.3f s" % t_one)
print("read ALL frames (eager)   : %7.3f s, %.0f MB resident" % (t_full, mem_mb))
print("lazy startup advantage    : %7.0f x" % (t_full / t_one))

## G) Write the movie config

Everything the movie step needs (except `PANELS` - see the note in section D)
is in `movie_settings`; the keys are exactly `make_frames.CONFIGURABLE`.
After writing the file, render the movie with:

```bash
cd ../Frames
mpirun -n 8 python make_frames.py movie_config.yaml
```

In [ ]:
MOVIE_CONFIG = os.path.abspath("../Frames/movie_config.yaml")

with open(MOVIE_CONFIG, "w") as f:
    yaml.safe_dump(movie_settings, f, sort_keys=False, default_flow_style=False)

print("wrote", MOVIE_CONFIG)
print()
print(open(MOVIE_CONFIG).read())